# Thêm Thư Viện

In [2]:
import pyodbc
import pandas as pd

# Tạo kết nối

In [3]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)
conn_dwh_lib = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=dwh_lib;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)


# ETL bảng Dim_Date

## Xóa data bảng DIM_date 

In [51]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Date"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ file CSV

In [ ]:
df_data_date = pd.read_csv("./Source-Data/Dim_Date.csv")
print(df_data_date)

       Date_key   Full_date                     Date_text   Day_name  \
0      19700101    1/1/1970     Thursday, January 1, 1970   Thursday   
1      19700102    1/2/1970       Friday, January 2, 1970     Friday   
2      19700103    1/3/1970     Saturday, January 3, 1970   Saturday   
3      19700104    1/4/1970       Sunday, January 4, 1970     Sunday   
4      19700105    1/5/1970       Monday, January 5, 1970     Monday   
...         ...         ...                           ...        ...   
29215  20491227  12/27/2049     Monday, December 27, 2049     Monday   
29216  20491228  12/28/2049    Tuesday, December 28, 2049    Tuesday   
29217  20491229  12/29/2049  Wednesday, December 29, 2049  Wednesday   
29218  20491230  12/30/2049   Thursday, December 30, 2049   Thursday   
29219  20491231  12/31/2049     Friday, December 31, 2049     Friday   

       Week_of_quarter  Day_of_week  Month  Quarter  Year  Day  
0                    1            4      1        1  1970    1  
1    

## Load data vào dwh_lib

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Date (Date_key, Full_date, Date_text, Day, Week_of_quarter, Month, Quarter, Year, Day_of_week, Day_name)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """
for index, row in df_data_date.iterrows():
    values = (row['Date_key'], 
              row['Full_date'], 
              row['Date_text'], 
              row['Day'], 
              row['Week_of_quarter'], 
              row['Month'], 
              row['Quarter'], 
              row['Year'], 
              row['Day_of_week'], 
              row['Day_name'])
    cursor_dwh.execute(insert_query, values)
    conn_dwh_lib.commit()

# ETL bảng Dim_Khoa

## Xóa data bảng DIM_Khoa

In [13]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Khoa"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ CSV

In [14]:
df_data_khoa = pd.read_csv("./Source-Data/Data_Khoa.csv")
print(df_data_khoa)

    ID_Khoa                             Ten_khoa
0         0                             Không rõ
1         1                    Lý luận Chính trị
2         2                    Khoa học ứng dụng
3         3                   Cơ khí Chế tạo máy
4         4                       Điện - Điện tử
5         5                      Cơ khí Động Lực
6         6                              Kinh tế
7         7                  Công nghệ thông tin
8         8                   In và Truyền thông
9         9          Công nghệ May và Thời Trang
10       10       Công nghệ Hóa học và Thực phẩm
11       11                             Xây dựng
12       12                            Ngoại ngữ
13       13               Đào tạo Chất lượng cao
14       14                Viện Sư phạm Kỹ thuật
15       15  Trường Trung học Kỹ thuật Thực hành


## Load data vào bảng Dim_Khoa

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Khoa (ID_khoa, Ten_khoa) 
                VALUES (?, ?)
                """
for index, row in df_data_khoa.iterrows():
    values = (row['ID_Khoa'], 
              row['Ten_khoa'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Dan_Toc

## Xóa data bảng DIM_Dan_Toc

In [167]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Dan_toc"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ CSV

In [ ]:
df_data_dantoc = pd.read_csv("./Source-Data/Data_Dim_Dan_toc.csv")
df_data_dantoc = df_data_dantoc.where(pd.notnull(df_data_dantoc), None)
print(df_data_dantoc)

    Mã               Tên                                       Tên gọi khác
0    1              Kinh                                               Việt
1    2               Tày          Thổ, Ngạn, Phén, Thù Lao, Pa Dí, Tày Khao
2    3              Thái  Tày Đăm, Tày Mười, Tày Thanh, Mán Thanh, Hàng ...
3    4               Hoa  Hán, Triều Châu, Phúc Kiến, Quảng Đông, Hải Na...
4    5            Khơ-me             Cur, Cul, Cu, Thổ, Việt gốc Miên, Krôm
5    6             Mường               Mol, Mual, Mọi, Mọi Bi, Ao Tá, Ậu Tá
6    7              Nùng  Xuồng, Giang, Nùng An, Phàn Sinh, Nùng Cháo, N...
7    8             HMông  Mèo, Hoa, Mèo Xanh, Mèo Đỏ, Mèo Đen, Ná Mẻo, M...
8    9               Dao  Mán, Động, Trại, Xá, Dìu, Miên, Kiềm, Miền, Qu...
9   10           Gia-rai    Giơ-rai, Tơ-buăn, Chơ-rai, Hơ-bau, Hđrung, Chor
10  11              Ngái                            Xín, Lê, Đản, Khách Gia
11  12              Ê-đê  Ra-đê, Đê, Kpạ, A-đham, Krung, Ktul, Đliê Ruê,...
12  13      

## Load data vào bảng DIM_Dan_toc

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Dan_toc (ID_dan_toc, Dan_toc, Ten_khac) 
                VALUES (?, ?, ?)
                """
for index, row in df_data_dantoc.iterrows():
    values = (row['Mã'], 
              row['Tên'], 
              row['Tên gọi khác'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Trinh_do

## Xóa data bảng Dim_Trinh_do

In [211]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Trinh_do"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [ ]:
query_trinhdo = "SELECT trinh_do_id, dbo.DecodeUTF8String(loai_trinh_do) AS Trinh_do FROM Trinh_do "
df_trinhdo = pd.read_sql(query_trinhdo, conn_libol)
print(df_trinhdo)

   trinh_do_id                 Trinh_do
0           10                   PGS.TS
1            3                 Cao đẳng
2            4                  Đại học
3            5                  Thạc sĩ
4            6                  Tiến sĩ
5            7              Phó tiến sĩ
6           11  Trung học chuyên nghiệp
7            9             Trung học PT
8           12                    12/12


C:\Users\admin\AppData\Local\Temp\ipykernel_17252\57140960.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Trinh_do = pd.read_sql(query_Trinh_do, conn_libol)


## Load data vào bảng Dim

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Trinh_do (ID_trinh_do, Loai_trinh_do) 
                VALUES (?, ?)
                """
for index, row in df_trinhdo.iterrows():
    values = (row['trinh_do_id'], 
              row['Trinh_do'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng DIM_Khoa_hoc

## Xóa data bảng DIM_Khoa_hoc

In [48]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Khoa_hoc"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ CSV

In [49]:
df_data_khoahoc = pd.read_csv("./Source-Data/Data_Dim_Khoa.csv")
print(df_data_khoahoc)

    ID_Khoa                             Ten_khoa
0         0                             Không rõ
1         1                    Lý luận Chính trị
2         2                    Khoa học ứng dụng
3         3                   Cơ khí Chế tạo máy
4         4                       Điện - Điện tử
5         5                      Cơ khí Động Lực
6         6                              Kinh tế
7         7                  Công nghệ thông tin
8         8                   In và Truyền thông
9         9          Công nghệ May và Thời Trang
10       10       Công nghệ Hóa học và Thực phẩm
11       11                             Xây dựng
12       12                            Ngoại ngữ
13       13               Đào tạo Chất lượng cao
14       14                Viện Sư phạm Kỹ thuật
15       15  Trường Trung học Kỹ thuật Thực hành


## Load data vào bảng Dim_Khoa

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Khoa_hoc (ID_khoa_hoc, Ten_khoa_hoc) 
                VALUES (?, ?)
                """
for index, row in df_data_khoahoc.iterrows():
    values = (row['ID khóa học'], 
              row['Tên khóa học'])
    cursor_dwh.execute(insert_query, values)   
conn_dwh_lib.commit()

# ETL bảng Dim_Lop

## Xóa data bảng Dim_Lop

In [50]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Lop"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [51]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Lop = "SELECT Ten_lop FROM Lop"
df_lop_lop = pd.read_sql(query_Lop, conn_libol)

# Đọc dữ liệu từ bảng Ban_doc trong CSDL libol
query_LopBandoc = "SELECT DISTINCT dbo.DecodeUTF8String(Lop) AS Lop FROM Ban_doc"
df_lop_bandoc = pd.read_sql(query_LopBandoc, conn_libol)

print(df_lop_lop)
print(df_lop_bandoc)

C:\Users\admin\AppData\Local\Temp\ipykernel_21948\869524976.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lop_lop = pd.read_sql(query_Lop, conn_libol)
C:\Users\admin\AppData\Local\Temp\ipykernel_21948\869524976.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lop_bandoc = pd.read_sql(query_LopBandoc, conn_libol)


       Ten_lop
0      191040B
1      191040A
2    19109CL1B
3    19109CL1A
4    19109CL2B
..         ...
597    211611B
598    211611A
599    211612A
600    211612B
601      21950

[602 rows x 1 columns]
            Lop
0        017031
1       001011C
2       042030A
3       057090A
4       117450A
...         ...
4259        709
4260   14151CLC
4261    181311A
4262  19143CL1B
4263      23950

[4264 rows x 1 columns]


## Xử lý data

In [53]:
# Tạo data_frame mới gộp các hàng dữ liệu từ 2 data_frame kia
df_data_lop = pd.DataFrame({"Ten_lop": pd.concat([df_lop_lop["Ten_lop"], 
                                                  df_lop_bandoc["Lop"]], 
                                                  ignore_index=True)})
df_data_lop['Ten_lop'] = df_data_lop['Ten_lop'].str.upper() # In hoa hết các hàng dữ liệu
for j, row in df_data_lop.iterrows():
    ten_nhom = row["Ten_lop"]
    if ((pd.isna(ten_nhom)) or # Kiểm tra none
        (ten_nhom == "") or
        (ten_nhom == "0") or
        (ten_nhom == "00") or
        (ten_nhom == "000")):  # Kiểm tra NaN
        df_data_lop.at[j, 'Ten_lop'] = "Không rõ"

df_data_lop = df_data_lop.sort_values(by="Ten_lop", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn
df_data_lop = df_data_lop.drop_duplicates().reset_index(drop=True) # xóa những hàng bị trùng nhau

df_data_lop['ID_lop'] = 0 # tạo ra cột ID đếm ID từ 0
df_data_lop['ID_khoa'] = "0" # cho ID_Khoa = 0 (Không rõ) vì hiện tại chưa có dữ liệu về lớp thuộc khoa nào 

for j, row in df_data_lop.iterrows(): # Cập nhật ID_lop từ 1 trở đi cho các dòng có Ten_lop khác "Không rõ"
    if row['Ten_lop'] != "Không rõ":
        df_data_lop.at[j, 'ID_lop'] = j + 1  # Gán ID_lop từ 1 trở đi

df_data_lop = df_data_lop.sort_values(by="ID_lop", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn
print(df_data_lop)

       Ten_lop  ID_lop ID_khoa
0     Không rõ       0       0
1       001011       1       0
2      001011A       2       0
3      001011C       3       0
4       001012       4       0
...        ...     ...     ...
4256  XÂY DỰNG    4257       0
4257   ÊN11021    4258       0
4258  ÊN14010A    4259       0
4259  ÊN2D02VD    4260       0
4260      ĐIỆN    4261       0

[4261 rows x 3 columns]


## Load data vào bảng Dim_Lop

In [54]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Lop (ID_lop, Ten_lop, ID_khoa) 
                VALUES (?, ?, ?)
                """
for index, row in df_data_lop.iterrows():
    values = (row['ID_lop'], 
              row['Ten_lop'], 
              row['ID_khoa'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Nhom_ban_doc

## Xóa data bảng Dim_Nhom_ban_doc

In [ ]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Nhom_ban_doc"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [27]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Nhombandoc = "SELECT Nhom_ID, dbo.DecodeUTF8String(Ten_nhom) AS Ten_nhom FROM Nhom_ban_doc"
df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_libol)
print(df_nhombandoc)

    Nhom_ID                  Ten_nhom
0         5                          
1         6         .Cán bộ công chức
2         9          MƯỢN & ĐỌC - SKV
3        10  Tốt nghiệp_Cộng Tác viên
4        12               Đọc tại chỗ
5        14                   MƯỢN GT
6        15             MƯỢN GT & SKV
7        16    Chưa tham gia khóa học
8        17    Con CB & CTV (Mượn GT)
9        18          HỌC VIÊN CAO HỌC
10       19    Khoa ĐT chất lượng cao
11       20         NHÓM NGOÀI TRƯỜNG
12       21       GIÁO TRÌNH QUÉT LỘN
13       22    NHÓM LÃNH ĐẠO, QUẢN LÝ
14       23         Nhóm ngoài trường
15       24      Nhóm ký công nợ (TN)
16       25           Nghiên cứu sinh


C:\Users\admin\AppData\Local\Temp\ipykernel_21948\1031425255.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_libol)


## Xử lý data

In [59]:
df_nhombandoc = df_nhombandoc.drop_duplicates().reset_index(drop=True) # xóa những hàng bị trùng nhau
for j, row in df_nhombandoc.iterrows():
    ten_nhom = row["Ten_nhom"]
    if pd.isna(ten_nhom) or ten_nhom == "":  # Kiểm tra NaN
        df_nhombandoc.at[j, 'Ten_nhom'] = "Không rõ"
    else:
        ten_nhom = ten_nhom[0].upper() + ten_nhom[1:].lower() # Chỉnh sửa chữ hoa chữ thường nếu chuỗi không rỗng
        df_nhombandoc.at[j, 'Ten_nhom'] = ten_nhom

print(df_nhombandoc)


    Nhom_ID                  Ten_nhom
0         5                  Không rõ
1         6         .cán bộ công chức
2         9          Mượn & đọc - skv
3        10  Tốt nghiệp_cộng tác viên
4        12               Đọc tại chỗ
5        14                   Mượn gt
6        15             Mượn gt & skv
7        16    Chưa tham gia khóa học
8        17    Con cb & ctv (mượn gt)
9        18          Học viên cao học
10       19    Khoa đt chất lượng cao
11       20         Nhóm ngoài trường
12       21       Giáo trình quét lộn
13       22    Nhóm lãnh đạo, quản lý
14       23         Nhóm ngoài trường
15       24      Nhóm ký công nợ (tn)
16       25           Nghiên cứu sinh


## Load data

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Nhom_ban_doc (ID_nhom_ban_doc, Nhom_ban_doc) 
                VALUES (?, ?)
                """
for index, row in df_nhombandoc.iterrows():
    values = (row['Nhom_ID'], 
              row['Ten_nhom'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Nhom_nghanh_nghe

## Xóa data bảng Dim_Nhom_nghanh_nghe

In [ ]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Nhom_nghanh_nghe"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [57]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Nhomnghanhnghe = "SELECT ID, dbo.DecodeUTF8String(Ten_nhom) AS Ten_nhom FROM Nhom_nghanh_nghe"
df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_libol)
print(df_nhomnghanhnge)

C:\Users\admin\AppData\Local\Temp\ipykernel_21948\443329784.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_libol)


      ID                  Ten_nhom
0      0          (Không xác định)
1      1                      Y tế
2      5       Công nhân viên chức
3      9         Điện tử - Tin học
4     11                 Tài chính
..   ...                       ...
248  291        Công nghệ vật liệu
249  292         Vật liệu xây dựng
250  293                      Luật
251  294         Sư phạm công nghệ
252  295  Kỹ thuật cơ khí động lực

[253 rows x 2 columns]


## Xử lý data

In [69]:
for j, row in df_nhomnghanhnge.iterrows(): 
    ten_nhom = row["Ten_nhom"]
    if pd.isna(ten_nhom) or ten_nhom == "":  # Kiểm tra none hoặc NaN
        df_nhombandoc.at[j, 'Ten_nhom'] = "(Không xác định)" 
df_nhomnghanhnge = df_nhomnghanhnge.drop_duplicates(subset='Ten_nhom').reset_index(drop=True) # xóa những hàng bị trùng nhau

print(df_nhomnghanhnge)

      ID                          Ten_nhom
0      0                  (Không xác định)
1      1                              Y tế
2      5               Công nhân viên chức
3      9                 Điện tử - Tin học
4     11                         Tài chính
..   ...                               ...
232  288  Logistic và Tài chính thương mại
233  292                 Vật liệu xây dựng
234  293                              Luật
235  294                 Sư phạm công nghệ
236  295          Kỹ thuật cơ khí động lực

[237 rows x 2 columns]


## Load data

In [62]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Nhom_nghanh_nghe (ID_nhom_nghanh_nghe, Nhom_nghanh_nghe) 
                VALUES (?, ?)
                """
for index, row in df_nhomnghanhnge.iterrows():
    values = (row['ID'], 
              row['Ten_nhom'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Quoc_gia

## Xóa data bảng Dim_Quoc_gia

In [ ]:
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Quoc_gia"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

## Đọc data từ SQL Server

In [66]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Quocgia = "SELECT Ma_nuoc_ID, Ma_ISO,Ten_nuoc_ISO FROM Ten_nuoc"
df_quocgia = pd.read_sql(query_Quocgia, conn_libol)
print(df_quocgia)

     Ma_nuoc_ID Ma_ISO         Ten_nuoc_ISO
0           190     TG                 Togo
1           191     TK              Tokelau
2           192     TO                Tonga
3           193     TT  Trinidad and Tobago
4           194     TN              Tunesia
..          ...    ...                  ...
231         186     SY                Syria
232         187     TW               Taiwan
233         188     TZ             Tanzania
234         189     TH             Thailand
235         209     VN              Vietnam

[236 rows x 3 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_21948\1487321348.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_quocgia = pd.read_sql(query_Quocgia, conn_libol)


## Xử lý data

In [70]:
df_quocgia = df_quocgia.sort_values(by="Ma_nuoc_ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn
df_quocgia = df_quocgia.drop_duplicates(subset='Ten_nuoc_ISO').reset_index(drop=True) # xóa những hàng bị trùng nhau

print(df_quocgia)

     Ma_nuoc_ID Ma_ISO        Ten_nuoc_ISO
0             1     AF         Afghanistan
1             2     AL             Albania
2             3     DZ             Algeria
3             4     AS      American Samoa
4             5     AD             Andorra
..          ...    ...                 ...
231         232     AJ          Azerbaijan
232         233     CI             Croatia
233         234     XV            Slovenia
234         235     BN  Bosnia-Hercegovina
235         236     XN           Macedonia

[236 rows x 3 columns]


## Load data

In [ ]:
cursor_dwh = conn_dwh_lib.cursor()
insert_query = """
                INSERT INTO DIM_Quoc_gia (ID_quoc_gia, Ma_ISO, Ten_nuoc_ISO) 
                VALUES (?, ?, ?)
                """
for index, row in df_quocgia.iterrows():
    values = (row['Ma_nuoc_ID'], 
              row['Ma_ISO'],
              row['Ten_nuoc_ISO'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_lib.commit()

# ETL bảng Dim_Vat_mang_tin

## Xóa data bảng 